# Curate Tools into Gateway Targets

The Module 3 Registry contains several MCP servers, but not every tool needs runtime governance. In this notebook you selectively promote the three tools the Travel Agent needs (`workshop-flights-mcp`, `workshop-hotels-mcp`, `workshop-search-knowledge-base`) into AgentCore Gateway targets.

This notebook is the boto3 equivalent of the **Curate Tools** CLI page. It discovers each Lambda's tool schemas, sanitizes them for the gateway, and creates one target per tool using an idempotent create-or-skip pattern.

## Why Selective Curation?

| Tool | Governance need | Gateway target? |
|------|-----------------|-----------------|
| Flights MCP | Audit + guardrails (pricing, PII) | Yes |
| Hotels MCP | Audit + guardrails (pricing, PII) | Yes |
| search-knowledge-base | Audit (enterprise data) | Yes |
| currenttime | None (low-risk utility) | No - Path A only |
| realserverfaketools | None (test fixture) | No - Path A only |

By the end of this notebook the gateway will have exactly three targets. The remaining Registry servers continue to serve traffic through Path A (NGINX) without the added overhead.

## Step 1: Set Up Variables

Look up the gateway ID (created in notebook 02) and the current account ID. You need both to build the Lambda target ARNs.

In [ ]:
import boto3, json

region = boto3.session.Session().region_name or "us-west-2"

sts = boto3.client("sts", region_name=region)
ACCOUNT_ID = sts.get_caller_identity()["Account"]

agentcore = boto3.client("bedrock-agentcore-control", region_name=region)
gateways = agentcore.list_gateways().get("items", [])
gw = [g for g in gateways if g["name"] == "tools-gateway"]
if not gw:
    raise RuntimeError(
        "Gateway 'tools-gateway' not found. Run notebook 02 first."
    )
GATEWAY_ID = gw[0]["gatewayId"]

print(f"Region:     {region}")
print(f"Account:    {ACCOUNT_ID}")
print(f"Gateway ID: {GATEWAY_ID}")

## Step 2: Define the Tool-to-Target Mapping

Map each Lambda function to the gateway target name the Sync Lambda expects (`tg-` prefix). These three names are the canonical target names used by notebooks 04 (sync verify) and 05 (test both paths), and by the Module 3a CLI walkthrough.

In [ ]:
TOOL_MAPPING = [
    ("workshop-flights-mcp", "tg-workshop-flights-mcp"),
    ("workshop-hotels-mcp", "tg-workshop-hotels-mcp"),
    ("workshop-search-knowledge-base", "tg-search-knowledge-base"),
]

for function_name, target_name in TOOL_MAPPING:
    print(f"  {function_name:35s} -> {target_name}")

## Step 3: Helper Functions - Discover, Sanitize, Create

Three helpers build the curation loop:

1. **`discover_tools`** invokes each Lambda with `tools/list` to retrieve its tool schemas.
2. **`clean_schema`** strips non-standard JSON Schema keys so the gateway's strict validator accepts the schema.
3. **`create_target_if_missing`** calls `create_gateway_target` with a `GATEWAY_IAM_ROLE` credential provider, skipping when the target already exists.

In [ ]:
lambda_client = boto3.client("lambda", region_name=region)

ALLOWED_SCHEMA_KEYS = {"type", "properties", "required", "items", "description"}


def clean_schema(s):
    """Strip fields the AgentCore Gateway does not accept from a JSON schema."""
    if not isinstance(s, dict):
        return {"type": "object"}
    cleaned = {}
    for k, v in s.items():
        if k not in ALLOWED_SCHEMA_KEYS:
            continue
        if k == "properties" and isinstance(v, dict):
            cleaned[k] = {pk: clean_schema(pv) for pk, pv in v.items()}
        elif k == "items":
            cleaned[k] = clean_schema(v)
        else:
            cleaned[k] = v
    return cleaned


def discover_tools(function_name):
    """Invoke a Lambda with tools/list and return its sanitized tool schemas."""
    resp = lambda_client.invoke(
        FunctionName=function_name,
        InvocationType="RequestResponse",
        Payload=json.dumps({"jsonrpc": "2.0", "method": "tools/list", "id": 1}),
    )
    if "FunctionError" in resp:
        raise RuntimeError(
            f"Lambda {function_name} returned FunctionError={resp['FunctionError']} "
            f"on tools/list: {resp['Payload'].read().decode('utf-8')[:300]}"
        )
    payload = json.loads(resp["Payload"].read().decode("utf-8"))
    if "error" in payload:
        raise RuntimeError(
            f"Lambda {function_name} returned JSON-RPC error: {payload['error']}"
        )
    tools = payload.get("result", {}).get("tools", [])
    for t in tools:
        t["inputSchema"] = clean_schema(t.get("inputSchema", {"type": "object"}))
    return tools


def create_target_if_missing(function_name, target_name, existing_names):
    """Create a gateway target for a Lambda MCP tool, or skip if it already exists."""
    if target_name in existing_names:
        print(f"  {target_name}: already exists, skipping")
        return "skipped"

    tools = discover_tools(function_name)
    if not tools:
        print(f"  {target_name}: no tools discovered from {function_name}, skipping")
        return "error"

    arn = f"arn:aws:lambda:{region}:{ACCOUNT_ID}:function:{function_name}"
    agentcore.create_gateway_target(
        gatewayIdentifier=GATEWAY_ID,
        name=target_name,
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": arn,
                    "toolSchema": {"inlinePayload": tools},
                }
            }
        },
        credentialProviderConfigurations=[
            {"credentialProviderType": "GATEWAY_IAM_ROLE"}
        ],
    )
    print(f"  {target_name}: created with {len(tools)} tool(s)")
    return "created"

## Step 4: Run the Curation Loop

Fetch the current list of gateway targets, then walk the mapping and create any missing ones. Re-running this cell is safe: every existing target is skipped.

In [ ]:
existing = agentcore.list_gateway_targets(gatewayIdentifier=GATEWAY_ID).get("items", [])
existing_names = {t["name"] for t in existing}

print(f"Existing targets before curation: {len(existing_names)}")
for n in sorted(existing_names):
    print(f"  - {n}")

print("\nCurating tools:")
results = {"created": 0, "skipped": 0, "error": 0}
for function_name, target_name in TOOL_MAPPING:
    outcome = create_target_if_missing(function_name, target_name, existing_names)
    results[outcome] += 1

print()
print(f"Summary: created={results['created']}  skipped={results['skipped']}  errors={results['error']}")
if results["error"] > 0:
    raise RuntimeError(
        f"{results['error']} gateway target(s) failed to create. "
        f"Check the log output above and re-run this cell."
    )

## Step 5: Verify Gateway Targets

List the targets one more time. You should see exactly the three curated targets (plus any targets the Sync Lambda has already created on its schedule). The CurrentTime and FakeTools servers remain on Path A only and do not appear here.

In [ ]:
targets = agentcore.list_gateway_targets(gatewayIdentifier=GATEWAY_ID).get("items", [])

expected = {tn for _, tn in TOOL_MAPPING}
present = {t["name"] for t in targets}

print(f"Gateway targets ({len(targets)} total):")
print("-" * 70)
for t in targets:
    name = t.get("name", "?")
    status = t.get("status", "?")
    marker = ">" if name in expected else " "
    print(f"{marker} {name:35s}  status={status}")

missing = expected - present
if missing:
    print(f"\nMissing expected targets: {sorted(missing)}")
else:
    print("\nAll expected targets are present.")

## Summary

You just promoted three tools from the Registry into AgentCore Gateway targets:

- `tg-workshop-flights-mcp`
- `tg-workshop-hotels-mcp`
- `tg-search-knowledge-base`

The remaining Registry servers (`currenttime`, `realserverfaketools`) stay on Path A via NGINX without audit or guardrails - that is the **selective curation** pattern in action. In the next notebook (`04-sync-catalog.ipynb`) you will run the Sync Lambda and see how it keeps the gateway aligned with the Registry on an EventBridge schedule.